In [1]:
# 정량적 평가 BLEU(기계번역 성능평가) ROUGE
# BLEU : 예측 문장과 정답 문장이 얼마나 많은 n-gram을 공유하는지 
# BLEU : unigram
# 단점 : 동의어를 인식 못함 (ex> The car is fast / the automobile is quick)

# ROUGH(문서요약 평가)
# n-gram을 비교.. Recall 중심 ( The cat sat on the mat / The cat sat )
# 정답 : 6, 예측 3
# recall : 6 / 3 = 0.5
# precision : 3 / 3 = 1 
# ROUGE-1 : unigram..

# 최근에는 다양한 평가지표가 사용 ( BERTScore, LLM-as-a-judge ... )

In [5]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
from kiwipiepy import Kiwi

kiwi = Kiwi()

# 한국어 형태소 단위 분리 함수
def tokenize_korean(text):
    return [token.form for token in kiwi.tokenize(text)]

# 스무딩 기법 점수가 불필요하게 0이 되는 문제를 완화하기위해 사용
def evaluate_ngram_improved(reference, candidate):
    smooth = SmoothingFunction().method4
    ref_tokens = [tokenize_korean(reference)]
    cand_tokens = tokenize_korean(candidate)

    bleu_score = sentence_bleu(ref_tokens, cand_tokens, smoothing_function=smooth)
    rouge = Rouge()
    ref_str = ' '.join(ref_tokens[0])
    cand_str = ' '.join(cand_tokens)
    scores = rouge.get_scores(cand_str, ref_str)[0]
    return bleu_score, scores

ref = '파리를 여행할 때는 에펠탑과 루브르 박물관을 꼭 방문해야 합니다.'
cand = '파리 여행 시 에펠탑과 루브르 박물관은 반드시 가봐야 할 명소입니다.'
bleu, rouge_rs = evaluate_ngram_improved(ref, cand)

print(f"BLEU : {bleu}")
print(f"ROUGE-1 F1 : {rouge_rs['rouge-1']['f']:.4f}")


BLEU : 0.194086232706272
ROUGE-1 F1 : 0.5882


In [9]:
# LM-Evaluation_Harness 벤치 마크 평가
# 모델의 추론 능력을 벤치마크 데이터 셋(hellaswag)을 통해 accuracy를 측정
import lm_eval
try:
    result = lm_eval.simple_evaluate(
        model="hf",
        model_args="pretrained=skt/kogpt2-base-v2,dtype=float32",
        tasks = ['kobest_hellaswag'],
        device='cpu',
        limit=2
    )
    print(f"LM-Eval 정확도(acc) : {result['results']['kobest_hellaswag']['acc,none']}")
except Exception as e:
    print(e)


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running loglikelihood requests: 100%|██████████| 8/8 [00:01<00:00,  6.95it/s]


LM-Eval 정확도(acc) : 0.0


In [ ]:
# 정답 형태가 고정되어 있지않은 생성형 태스킈의 경우 사용모델을 판단기준으로 활용
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI()
def evaluate_with_gpt(prompt, generated_text):
    eval_prompt = f'''
다음 질문과 답변을 보고 정확성, 유창성, 관련성을 기준으로 1~5점 사이의 점수와 이유를 작성해주세요
질문:{prompt}
답변:{generated_text}
'''
    try:
        response = client.chat.completions.create(
            model = 'gpt-5.4-nano',
            messages=[{'role':'user', 'content':eval_prompt}],
            max_completion_tokens = 150
        )
        return response.choices[0].message.content
    except Exception as e:
        return str(e)

evaluate_with_gpt('프랑스의 명소는?', '파리 여행시 에펠탑과 루브르 박물관은 반드시 가봐야할 명소 입니다.')


'다음은 요청하신 기준(정확성, 유창성, 관련성)에 따른 평가입니다.\n\n## 1) 정확성: 5/5  \n- 에펠탑과 루브르 박물관은 프랑스(특히 파리)의 대표적인 명소로 널리 알려져 있으며, 사실과도 부합합니다.\n\n## 2) 유창성: 4/5  \n- 문장이 자연스럽고 이해하기 쉽습니다.  \n- 다만 “프랑스의 명소는?”이라는 질문에 대해 “파리 여행시”라고 범위를 조금 더 구체화했으면 더 매끄러울 수 있습니다(프랑스 전체가'